# Depth + Structure Guided Real-ESRGAN

한 장의 입력 이미지에 대해 다음 4가지를 비교합니다.

1. Original input
2. Lanczos x4 upsampling
3. Real-ESRGAN x4
4. Proposed: Depth + Structure 기반 adaptive fusion

최종 출력:

`I_final = W * I_sr + (1 - W) * I_up`

- 가까운 영역: Real-ESRGAN 비중을 높게 유지
- 가까운 + 구조적 영역: 거의 Real-ESRGAN 100%
- 매우 가까운 영역: 구조 여부와 무관하게 Real-ESRGAN 100%
- 먼 영역이라도 구조가 강하면 Real-ESRGAN 비중을 일정 수준 유지


## 실행 방법

이 노트북은 **Depth Anything V2 Small (Hugging Face) + Real-ESRGAN x4plus**를 사용합니다.

셀을 위에서 아래로 순서대로 실행하세요. NumPy/SciPy/pycolmap/Depth Anything 3는 사용하지 않습니다.


In [ ]:
import sys
sys.path.insert(0, "/content/Real-ESRGAN")

In [ ]:
!pip install -U setuptools wheel cython
%cd /content

!rm -rf BasicSR
!git clone https://github.com/XPixelGroup/BasicSR.git

In [ ]:
import sys
import types
import torchvision.transforms.functional as F

module_name = "torchvision.transforms.functional_tensor"

if module_name not in sys.modules:
    mod = types.ModuleType(module_name)
    mod.rgb_to_grayscale = F.rgb_to_grayscale
    sys.modules[module_name] = mod

In [ ]:
import sys

sys.path.insert(0, "/content/BasicSR")
sys.path.insert(0, "/content/Real-ESRGAN")

print(sys.path[:3])
from pathlib import Path

version_file = Path("/content/BasicSR/basicsr/version.py")

version_file.write_text(
    """
__version__ = "1.4.2"
__gitsha__ = "unknown"
version_info = (1, 4, 2)
""".strip()
)

print("created:", version_file)

from basicsr.archs.rrdbnet_arch import RRDBNet

print("BasicSR import OK")

In [ ]:
import sys
import types
import torchvision.transforms.functional as F

module_name = "torchvision.transforms.functional_tensor"

if module_name not in sys.modules:
    mod = types.ModuleType(module_name)
    mod.rgb_to_grayscale = F.rgb_to_grayscale
    sys.modules[module_name] = mod

print("torchvision compatibility patch OK")

In [ ]:
import sys

sys.path.insert(0, "/content/BasicSR")
sys.path.insert(0, "/content/Real-ESRGAN")

print(sys.path[:3])
from pathlib import Path

In [ ]:
# ============================================================
# Real-ESRGAN local repo를 현재 Python 환경에 등록
# dependencies는 설치하지 않음
# ============================================================

# ============================================================
# 1. Real-ESRGAN 저장소만 복구
# ============================================================
import subprocess
import os
REPO = "/content/Real-ESRGAN"

if not os.path.isdir(REPO):

    print("Real-ESRGAN repo 없음 -> clone")

    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/xinntao/Real-ESRGAN.git",
            REPO
        ],
        check=True
    )

else:
    print("Real-ESRGAN repo 이미 존재")

%cd /content/Real-ESRGAN

!python -m pip install -e . --no-deps

# ============================================================
# Import cache 초기화 + 확인
# ============================================================

import sys
import importlib
import types

importlib.invalidate_caches()

# 혹시 이전 실패 import 흔적 제거
for name in list(sys.modules.keys()):
    if name == "realesrgan" or name.startswith("realesrgan."):
        del sys.modules[name]


# torchvision compatibility patch
import torchvision.transforms.functional as TF

module_name = "torchvision.transforms.functional_tensor"

mod = types.ModuleType(module_name)
mod.rgb_to_grayscale = TF.rgb_to_grayscale
sys.modules[module_name] = mod


# ============================================================
# 실제 import 위치 확인
# ============================================================

spec = importlib.util.find_spec("realesrgan")

print("realesrgan spec:", spec)

if spec is not None:
    print("origin:", spec.origin)
    print("locations:", spec.submodule_search_locations)


# ============================================================
# import
# ============================================================

from basicsr.archs.rrdbnet_arch import RRDBNet
print("BasicSR OK")

from realesrgan import RealESRGANer
print("Real-ESRGAN OK")

In [ ]:
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

print("BasicSR OK")
print("Real-ESRGAN OK")

In [ ]:
import torch
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

%cd /content/Real-ESRGAN

!mkdir -p weights

!wget -O weights/realesrgan-x4plus.pth \
https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth
# ==============================
# 1. RRDBNet 모델 정의
# ==============================

model = RRDBNet(
    num_in_ch=3,
    num_out_ch=3,
    num_feat=64,
    num_block=23,
    num_grow_ch=32,
    scale=4
)

# ==============================
# 2. Real-ESRGAN 로더
# ==============================

upsampler = RealESRGANer(
    scale=4,
    model_path="/content/Real-ESRGAN/weights/realesrgan-x4plus.pth",
    model=model,
    tile=0,
    tile_pad=10,
    pre_pad=0,
    half=torch.cuda.is_available()
)

print("Real-ESRGAN model ready")

In [ ]:
# ============================================================
# Depth Anything V2 Small - Hugging Face
# 이미지 1장 -> Depth Map
# ============================================================

!pip install -q -U transformers

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
from google.colab import files


# ============================================================
# 1. Device
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)


# ============================================================
# 2. Depth Anything V2 Small 로드
# ============================================================

MODEL_ID = "depth-anything/Depth-Anything-V2-Small-hf"

print("Depth Anything V2 Small 로드 중...")

processor = AutoImageProcessor.from_pretrained(MODEL_ID)

model = AutoModelForDepthEstimation.from_pretrained(
    MODEL_ID
).to(device).eval()

print("모델 로드 완료")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# DIV2K Fine-tuning — Proposed Depth + Structure Pipeline

기존 **Depth + Structure Guided Real-ESRGAN** 아이디어를 그대로 유지하되, 이제 최종 `Proposed` 출력에 직접 L1 loss를 걸어 RRDBNet을 업데이트합니다.

학습 경로:

`LR → RRDBNet(Real-ESRGAN) → Depth/Structure → Depth-guided Blur + Structure Protection → Proposed → L1(Proposed, HR)`

- DIV2K: **train 720 / validation 80 / test 100**
- Optimizer: **Adam**
- Learning Rate: **1e-5**
- Weight Decay: **1e-6**
- Loss: **L1 Loss**
- Max Epoch: **30**
- Scheduler: **ReduceLROnPlateau**
- Early Stopping: **Validation PSNR 4회 연속 미개선 시 종료**
- Gradient Clipping: **1.0**
- Mixed Precision: **AMP**

> Depth Anything V2는 conditioning map 생성용으로 freeze합니다. 역전파는 최종 Proposed 출력에서 RRDBNet까지 흐릅니다. 기존 아래쪽 단일 이미지 실험/시각화 코드는 그대로 사용할 수 있도록, 학습 완료 후 `upsampler`만 best checkpoint로 다시 연결합니다.


In [ ]:
# 1. IMPORT / SEED
import os, cv2, math, glob, time, random, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
!mkdir -p /content/DIV2K

!wget -P /content/DIV2K \
http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip

!unzip -q /content/DIV2K/DIV2K_train_HR.zip -d /content/DIV2K

In [ ]:
# ============================================================
# DIV2K Validation HR 100장
# GitHub sparse checkout 방식
# ============================================================

import os
import glob
import shutil
import subprocess

DIV2K_ROOT = "/content/DIV2K"
VALID_HR_DIR = os.path.join(DIV2K_ROOT, "DIV2K_valid_HR")

os.makedirs(DIV2K_ROOT, exist_ok=True)

valid_files = glob.glob(os.path.join(VALID_HR_DIR, "*"))

if len(valid_files) != 100:
    print("DIV2K Validation HR 다운로드 중...")

    repo_dir = "/content/div2k_github"

    if os.path.exists(repo_dir):
        shutil.rmtree(repo_dir)

    # 저장소 전체 blob을 받지 않고 필요한 폴더만 checkout
    subprocess.run([
        "git",
        "clone",
        "--depth", "1",
        "--filter=blob:none",
        "--sparse",
        "https://github.com/Benjamin-Wegener/DIV2K.git",
        repo_dir
    ], check=True)

    subprocess.run([
        "git",
        "-C", repo_dir,
        "sparse-checkout",
        "set",
        "DIV2K_valid_HR"
    ], check=True)

    src_dir = os.path.join(repo_dir, "DIV2K_valid_HR")

    if not os.path.isdir(src_dir):
        raise RuntimeError("GitHub에서 DIV2K_valid_HR 폴더를 찾지 못했습니다.")

    # 기존 폴더 초기화
    if os.path.exists(VALID_HR_DIR):
        shutil.rmtree(VALID_HR_DIR)

    shutil.copytree(src_dir, VALID_HR_DIR)

    # 임시 저장소 삭제
    shutil.rmtree(repo_dir)

# 확인
valid_files = sorted(
    glob.glob(os.path.join(VALID_HR_DIR, "*"))
)

print("Validation HR:", len(valid_files))

assert len(valid_files) == 100, (
    f"Validation HR 이미지 수 오류: {len(valid_files)}"
)

print("DIV2K Validation 준비 완료")

In [ ]:
# ============================================================
# FT-1. DIV2K 경로 확인 + Train / Val / Test split
#
# 기대 폴더:
# /content/DIV2K/DIV2K_train_HR  -> 800장
# /content/DIV2K/DIV2K_valid_HR  -> 100장
#
# Validation 폴더는 .png/.jpg/.jpeg 모두 허용
# ============================================================

import os
import glob
import random
import time
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

TRAIN_HR_DIR = "/content/DIV2K/DIV2K_train_HR"
TEST_HR_DIR  = "/content/DIV2K/DIV2K_valid_HR"

def get_image_paths(folder):
    paths = []
    for ext in ["*.png", "*.jpg", "*.jpeg", "*.PNG", "*.JPG", "*.JPEG"]:
        paths.extend(glob.glob(os.path.join(folder, ext)))
    return sorted(set(paths))

train_all = get_image_paths(TRAIN_HR_DIR)
test_paths = get_image_paths(TEST_HR_DIR)

print("DIV2K train HR:", len(train_all))
print("DIV2K hold-out test HR:", len(test_paths))

if len(train_all) != 800:
    raise RuntimeError(
        f"DIV2K train HR 800장이 필요합니다. 현재: {len(train_all)}"
    )

if len(test_paths) != 100:
    raise RuntimeError(
        f"DIV2K validation HR 100장이 필요합니다. 현재: {len(test_paths)}"
    )

# train 800 -> train 720 / validation 80
rng = np.random.default_rng(SEED)
indices = np.arange(len(train_all))
rng.shuffle(indices)

val_count = 80
val_idx = set(indices[:val_count].tolist())

train_paths = [
    p for i, p in enumerate(train_all)
    if i not in val_idx
]

val_paths = [
    p for i, p in enumerate(train_all)
    if i in val_idx
]

print("Train:", len(train_paths))
print("Val  :", len(val_paths))
print("Test :", len(test_paths))

assert len(train_paths) == 720
assert len(val_paths) == 80
assert len(test_paths) == 100


In [ ]:
# 3. Train / Val / Test split
rng = np.random.default_rng(SEED)
indices = np.arange(len(train_all))
rng.shuffle(indices)

val_count = 80
val_idx = set(indices[:val_count].tolist())

train_paths = [p for i,p in enumerate(train_all) if i not in val_idx]
val_paths = [p for i,p in enumerate(train_all) if i in val_idx]

assert len(train_paths) == 720
assert len(val_paths) == 80
assert len(test_paths) == 100

print("Train:", len(train_paths))
print("Val:", len(val_paths))
print("Test:", len(test_paths))


In [ ]:
from pathlib import Path

DIV2K_ROOT = Path("/content/DIV2K")

TRAIN_HR_DIR = DIV2K_ROOT / "DIV2K_train_HR"
VALID_HR_DIR = DIV2K_ROOT / "DIV2K_valid_HR"


def get_image_files(folder):
    exts = {".png", ".jpg", ".jpeg"}

    return sorted([
        p for p in Path(folder).iterdir()
        if p.suffix.lower() in exts
    ])


# DIV2K train HR: 800장
all_train_files = get_image_files(TRAIN_HR_DIR)

# DIV2K valid HR: 100장
all_test_files = get_image_files(VALID_HR_DIR)

print("DIV2K train HR:", len(all_train_files))
print("DIV2K valid HR:", len(all_test_files))

assert len(all_train_files) == 800, \
    f"train HR 개수 오류: {len(all_train_files)}"

assert len(all_test_files) == 100, \
    f"valid HR 개수 오류: {len(all_test_files)}"


# ============================================================
# 720 / 80 / 100 분할
# ============================================================

TRAIN_FILES = all_train_files[:720]
VAL_FILES   = all_train_files[720:800]
TEST_FILES  = all_test_files

print(
    f"train={len(TRAIN_FILES)}, "
    f"val={len(VAL_FILES)}, "
    f"test={len(TEST_FILES)}"
)

In [ ]:
# ============================================================
# 2. DIV2K Dataset / DataLoader
#    HR patch -> bicubic x4 downsample -> LR
# ============================================================

import math
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms.functional as TF

SCALE = 4
TRAIN_HR_PATCH = 256     # Colab GPU 메모리에 맞춘 patch 학습
EVAL_HR_PATCH = 256      # validation/test도 모든 이미지에서 동일 크기 crop 평가
BATCH_SIZE = 2
NUM_WORKERS = 2
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

class DIV2KPairDataset(Dataset):
    def __init__(self, files, hr_patch=256, train=True, scale=4):
        self.files = list(files)
        self.hr_patch = hr_patch
        self.train = train
        self.scale = scale

    def __len__(self):
        return len(self.files)

    def _crop(self, img):
        w, h = img.size
        p = self.hr_patch

        if w < p or h < p:
            new_w = max(w, p)
            new_h = max(h, p)
            img = img.resize((new_w, new_h), Image.Resampling.BICUBIC)
            w, h = img.size

        if self.train:
            left = random.randint(0, w - p)
            top  = random.randint(0, h - p)
        else:
            left = (w - p) // 2
            top  = (h - p) // 2

        return img.crop((left, top, left + p, top + p))

    def __getitem__(self, idx):
        hr = Image.open(self.files[idx]).convert('RGB')
        hr = self._crop(hr)

        # 학습 데이터 augmentation은 아이디어를 바꾸지 않는 범위에서 flip만 사용
        if self.train:
            if random.random() < 0.5:
                hr = TF.hflip(hr)
            if random.random() < 0.5:
                hr = TF.vflip(hr)

        hr = TF.to_tensor(hr)  # [0, 1], RGB

        lr = TF.resize(
            hr,
            [self.hr_patch // self.scale, self.hr_patch // self.scale],
            interpolation=TF.InterpolationMode.BICUBIC,
            antialias=True,
        )

        return lr, hr, self.files[idx].name

train_dataset = DIV2KPairDataset(TRAIN_FILES, TRAIN_HR_PATCH, train=True, scale=SCALE)
val_dataset   = DIV2KPairDataset(VAL_FILES,   EVAL_HR_PATCH,  train=False, scale=SCALE)
test_dataset  = DIV2KPairDataset(TEST_FILES,  EVAL_HR_PATCH,  train=False, scale=SCALE)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print('Dataset / DataLoader ready')
print(f'train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}')
print('train batches:', len(train_loader))


In [ ]:
import os
import glob

paths = [
    "/content/DIV2K/DIV2K_train_HR",
    "/content/DIV2K/DIV2K_valid_HR",
]

for p in paths:
    print("\nPATH:", p)
    print("exists:", os.path.exists(p))
    print("png count:", len(glob.glob(os.path.join(p, "*.png"))))

    if os.path.exists(p):
        print("first files:", os.listdir(p)[:5])

In [ ]:
import glob
import os

train_hr = sorted(
    glob.glob("/content/DIV2K/DIV2K_train_HR/*.png")
)

valid_hr = sorted(
    glob.glob("/content/DIV2K/DIV2K_valid_HR/*.png")
    + glob.glob("/content/DIV2K/DIV2K_valid_HR/*.jpg")
    + glob.glob("/content/DIV2K/DIV2K_valid_HR/*.jpeg")
)

print("Train HR :", len(train_hr))
print("Valid HR :", len(valid_hr))

train_files = train_hr[:720]
val_files   = train_hr[720:800]
test_files  = valid_hr

print(
    f"train={len(train_files)}, "
    f"val={len(val_files)}, "
    f"test={len(test_files)}"
)

In [ ]:
# ============================================================
# 3. 기존 아이디어를 PyTorch로 구현
#    최종 Proposed 출력까지 differentiable하게 연결
#    ★ AMP 안정화: RRDBNet만 AMP, Depth/Mask/Fusion은 FP32
# ============================================================

import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TVF
from PIL import Image

# cell 11에서 로드한 Depth Anything V2 model을 별도 이름으로 보존
depth_model = model
depth_model.eval()
for p in depth_model.parameters():
    p.requires_grad_(False)

# ---- 기존 ONE-CELL EXPERIMENT 파라미터 유지 ----
DEPTH_FAR = 0.10
DEPTH_NEAR = 0.30

EDGE_WEIGHT = 0.4
RESIDUAL_WEIGHT = 0.6
RESIDUAL_PERCENTILE = 80

STRUCTURE_START = 0.15
STRUCTURE_PROTECT_POWER = 0.4

MIN_STRUCTURE_BLUR = 0.03
BLUR_SIGMA = 2.5
MASK_SIGMA = 1.0


def _odd_kernel_from_sigma(sigma, minimum=3):
    k = max(minimum, int(2 * round(3 * sigma) + 1))
    return k if k % 2 == 1 else k + 1


def gaussian_blur_tensor(x, sigma):
    if sigma <= 0:
        return x
    k = _odd_kernel_from_sigma(sigma)
    return TVF.gaussian_blur(x, [k, k], [sigma, sigma])


def _check_finite(name, x):
    """NaN/Inf가 처음 생기는 지점을 정확히 표시."""
    if not torch.isfinite(x).all():
        finite = x[torch.isfinite(x)]
        if finite.numel() > 0:
            mn = finite.min().item()
            mx = finite.max().item()
        else:
            mn = float("nan")
            mx = float("nan")
        n_nan = torch.isnan(x).sum().item()
        n_inf = torch.isinf(x).sum().item()
        raise RuntimeError(
            f"[{name}] NaN/Inf detected | "
            f"nan={n_nan}, inf={n_inf}, finite_min={mn}, finite_max={mx}"
        )


def depth_map_from_lr(lr, out_size):
    """
    Depth Anything V2는 conditioning map 생성 전용.
    바깥 autocast와 무관하게 반드시 FP32로 실행.
    반환: [B,1,H,W], 0=far / 1=near
    """
    pil_images = []
    lr_cpu = lr.detach().float().cpu().clamp(0.0, 1.0)

    for img in lr_cpu:
        arr = (
            img.permute(1, 2, 0).numpy() * 255.0
        ).round().astype(np.uint8)
        # Pillow 13 대비: mode='RGB' 제거
        pil_images.append(Image.fromarray(arr))

    inputs = processor(images=pil_images, return_tensors="pt")

    # floating tensor는 FP32, 그 외 tensor는 dtype 유지
    inputs = {
        k: (
            v.to(device=device, dtype=torch.float32)
            if torch.is_floating_point(v)
            else v.to(device)
        )
        for k, v in inputs.items()
    }

    with torch.no_grad():
        # ★ 외부 AMP가 켜져 있어도 Depth Anything은 FP32
        with torch.amp.autocast(device_type="cuda", enabled=False):
            pred = depth_model(**inputs).predicted_depth
            pred = pred.unsqueeze(1).float()
            pred = F.interpolate(
                pred,
                size=out_size,
                mode="bicubic",
                align_corners=False,
            )

            _check_finite("depth_raw", pred)

            flat = pred.flatten(1)
            low = torch.quantile(
                flat, 0.02, dim=1
            ).view(-1, 1, 1, 1)
            high = torch.quantile(
                flat, 0.98, dim=1
            ).view(-1, 1, 1, 1)

            # 1e-8 대신 FP32에서 충분히 안전한 하한
            denom = (high - low).clamp_min(1e-6)

            depth = (
                (pred - low) / denom
            ).clamp(0.0, 1.0)

            _check_finite("depth_norm", depth)

    return depth


def make_edge_structure_map_torch(img_rgb, blur_sigma=1.5):
    img_rgb = img_rgb.float()

    gray = (
        0.2989 * img_rgb[:, 0:1]
        + 0.5870 * img_rgb[:, 1:2]
        + 0.1140 * img_rgb[:, 2:3]
    )

    kx = torch.tensor(
        [
            [-1., 0., 1.],
            [-2., 0., 2.],
            [-1., 0., 1.],
        ],
        device=img_rgb.device,
        dtype=torch.float32,
    ).view(1, 1, 3, 3)

    ky = kx.transpose(2, 3)

    gx = F.conv2d(gray, kx, padding=1)
    gy = F.conv2d(gray, ky, padding=1)

    mag = torch.sqrt(
        gx.square() + gy.square() + 1e-12
    )

    norm = torch.quantile(
        mag.detach().flatten(1),
        0.99,
        dim=1,
    ).view(-1, 1, 1, 1)

    norm = norm.clamp_min(1e-6)

    mag = (
        mag / norm
    ).clamp(0.0, 1.0)

    out = gaussian_blur_tensor(
        mag, blur_sigma
    ).clamp(0.0, 1.0)

    _check_finite("edge_map", out)
    return out


def make_residual_structure_map_torch(
    I_up,
    I_sr,
    percentile=80,
    blur_sigma=2.0,
):
    I_up = I_up.float()
    I_sr = I_sr.float()

    diff = torch.abs(
        I_sr - I_up
    ).mean(dim=1, keepdim=True)

    _check_finite("residual_diff", diff)

    q = percentile / 100.0
    flat = diff.detach().flatten(1)

    threshold = torch.quantile(
        flat, q, dim=1
    ).view(-1, 1, 1, 1)

    high = torch.quantile(
        flat, 0.99, dim=1
    ).view(-1, 1, 1, 1)

    denom = (
        high - threshold
    ).clamp_min(1e-6)

    residual = (
        (diff - threshold) / denom
    ).clamp(0.0, 1.0)

    residual = gaussian_blur_tensor(
        residual, blur_sigma
    ).clamp(0.0, 1.0)

    _check_finite("residual_map", residual)
    return residual


def proposed_forward(sr_model, lr, return_maps=False):
    """
    RRDBNet convolution은 AMP 사용.
    Depth / quantile / structure / blur / fusion은 FP32.
    최종 I_final에서 RRDBNet까지 gradient가 흐름.
    """

    # --------------------------------------------------------
    # 1. RRDBNet만 AMP
    # --------------------------------------------------------
    with torch.amp.autocast(
        device_type="cuda",
        enabled=AMP_ENABLED,
    ):
        I_sr = sr_model(lr)

    _check_finite("rrdbnet_output", I_sr)

    # 여기부터 FP32
    I_sr_fp32 = I_sr.float()

    I_up = F.interpolate(
        lr.float(),
        scale_factor=SCALE,
        mode="bicubic",
        align_corners=False,
    )

    _check_finite("bicubic_up", I_up)

    # --------------------------------------------------------
    # 2. Depth
    # --------------------------------------------------------
    M_depth = depth_map_from_lr(
        lr,
        I_sr.shape[-2:],
    ).to(
        device=I_sr.device,
        dtype=torch.float32,
    )

    # --------------------------------------------------------
    # 3. Edge
    # --------------------------------------------------------
    M_edge = make_edge_structure_map_torch(
        I_up,
        blur_sigma=1.5,
    )

    # --------------------------------------------------------
    # 4. Residual
    # --------------------------------------------------------
    M_residual = make_residual_structure_map_torch(
        I_up,
        I_sr_fp32,
        percentile=RESIDUAL_PERCENTILE,
        blur_sigma=2.0,
    )

    S = (
        EDGE_WEIGHT * M_edge
        + RESIDUAL_WEIGHT * M_residual
    ).clamp(0.0, 1.0)

    _check_finite("structure_map", S)

    # --------------------------------------------------------
    # 5. Depth -> far
    # --------------------------------------------------------
    D = M_depth.clamp(0.0, 1.0)

    far_strength = 1.0 - (
        (D - DEPTH_FAR)
        / max(DEPTH_NEAR - DEPTH_FAR, 1e-6)
    ).clamp(0.0, 1.0)

    far_strength = (
        far_strength
        * far_strength
        * (3.0 - 2.0 * far_strength)
    )

    _check_finite("far_strength", far_strength)

    # --------------------------------------------------------
    # 6. Structure protection
    # --------------------------------------------------------
    structure_raw = (
    (S - STRUCTURE_START)
    / max(1.0 - STRUCTURE_START, 1e-6)
    ).clamp(0.0, 1.0)

    POWER_EPS = 1e-4

    eps_pow = POWER_EPS ** STRUCTURE_PROTECT_POWER

    structure_protect = (
        (structure_raw + POWER_EPS).pow(
            STRUCTURE_PROTECT_POWER
        )
        - eps_pow
    )

    structure_protect = structure_protect / (
        (1.0 + POWER_EPS) ** STRUCTURE_PROTECT_POWER
        - eps_pow
    )

    structure_protect = structure_protect.clamp(
        0.0,
        1.0
    )

    _check_finite(
        "structure_protect",
        structure_protect
    )

    _check_finite(
        "structure_protect",
        structure_protect,
    )

    # --------------------------------------------------------
    # 7. Blur strength
    # --------------------------------------------------------
    blur_strength = far_strength * (
        1.0
        - (1.0 - MIN_STRUCTURE_BLUR)
        * structure_protect
    )

    blur_strength = gaussian_blur_tensor(
        blur_strength,
        MASK_SIGMA,
    ).clamp(0.0, 1.0)

    _check_finite("blur_strength", blur_strength)

    # --------------------------------------------------------
    # 8. Proposed final
    # --------------------------------------------------------
    I_blur = gaussian_blur_tensor(
        I_sr_fp32,
        BLUR_SIGMA,
    )

    _check_finite("blurred_sr", I_blur)

    I_final = (
        (1.0 - blur_strength) * I_sr_fp32
        + blur_strength * I_blur
    )

    _check_finite("I_final", I_final)

    if return_maps:
        maps = {
            "I_sr": I_sr_fp32,
            "I_up": I_up,
            "depth": D,
            "edge": M_edge,
            "residual": M_residual,
            "structure": S,
            "far_strength": far_strength,
            "structure_protect": structure_protect,
            "blur_strength": blur_strength,
        }
        return I_final, maps

    return I_final


print("Differentiable Proposed pipeline ready (AMP-safe)")


In [ ]:
# ============================================================
# 4. Real-ESRGAN x4plus pretrained weight -> Fine-tuning 준비
# ============================================================

import os
import torch
import torch.nn as nn
from basicsr.archs.rrdbnet_arch import RRDBNet

TRAIN_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Training device:', TRAIN_DEVICE)

sr_train_model = RRDBNet(
    num_in_ch=3,
    num_out_ch=3,
    num_feat=64,
    num_block=23,
    num_grow_ch=32,
    scale=4,
).to(TRAIN_DEVICE)

PRETRAINED_PATH = '/content/Real-ESRGAN/weights/realesrgan-x4plus.pth'
pretrained = torch.load(PRETRAINED_PATH, map_location='cpu')

if 'params_ema' in pretrained:
    pretrained_state = pretrained['params_ema']
elif 'params' in pretrained:
    pretrained_state = pretrained['params']
else:
    pretrained_state = pretrained

sr_train_model.load_state_dict(pretrained_state, strict=True)
print('Loaded pretrained Real-ESRGAN x4plus weights')

# ---- 요청한 학습 설정 ----
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-6
MAX_EPOCH = 30
GRAD_CLIP = 1.0
EARLY_STOP_PATIENCE = 4

optimizer = torch.optim.Adam(
    sr_train_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

criterion = nn.L1Loss()

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',          # Validation PSNR가 높을수록 좋음
    factor=0.5,
    patience=1,
    threshold=1e-4,
    min_lr=1e-7,
)

AMP_ENABLED = torch.cuda.is_available()
scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)

CHECKPOINT_DIR = '/content/drive/MyDrive/KSY/capstone/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
BEST_PATH = os.path.join(CHECKPOINT_DIR, 'best_proposed_realesrgan_div2k.pth')

print('Optimizer       : Adam')
print('LR              :', LEARNING_RATE)
print('Weight decay    :', WEIGHT_DECAY)
print('Loss            : L1Loss(Proposed, HR)')
print('Max epoch       :', MAX_EPOCH)
print('Scheduler       : ReduceLROnPlateau(val PSNR)')
print('Early stopping  :', EARLY_STOP_PATIENCE, 'epochs without val PSNR improvement')
print('Gradient clip   :', GRAD_CLIP)
print('AMP             :', AMP_ENABLED)
print('Best checkpoint :', BEST_PATH)


In [ ]:
# ============================================================
# AMP GradScaler 재생성
# 학습 셀 실행 직전에 실행
# ============================================================

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=AMP_ENABLED
)

print("GradScaler reset")
print("scale:", scaler.get_scale())

In [ ]:
# ============================================================
# 5. Train / Validation
#    ★ Loss는 RRDBNet 원본 출력이 아니라 Proposed final에 적용
#    ★ RRDBNet 내부만 AMP, L1/PSNR은 FP32
# ============================================================

from tqdm.auto import tqdm
import math


def batch_psnr(pred, target):
    pred = pred.float().clamp(0.0, 1.0)
    target = target.float().clamp(0.0, 1.0)

    mse = (
        (pred - target)
        .pow(2)
        .flatten(1)
        .mean(dim=1)
    )

    psnr = 10.0 * torch.log10(
        1.0 / (mse + 1e-12)
    )
    return psnr


@torch.no_grad()
def evaluate_proposed(
    sr_model,
    loader,
    desc="Validation",
):
    sr_model.eval()

    loss_sum = 0.0
    psnr_sum = 0.0
    count = 0

    pbar = tqdm(
        loader,
        desc=desc,
        leave=False,
    )

    for lr, hr, _ in pbar:
        lr = lr.to(
            TRAIN_DEVICE,
            non_blocking=True,
        )
        hr = hr.to(
            TRAIN_DEVICE,
            non_blocking=True,
        )

        # proposed_forward 내부에서 RRDBNet만 AMP 처리
        proposed = proposed_forward(
            sr_model,
            lr,
        )

        # loss / metric은 FP32
        loss = criterion(
            proposed.float(),
            hr.float(),
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                "Validation loss became NaN/Inf"
            )

        # ★ 기존 pred 오타 수정
        psnr_values = batch_psnr(
            proposed,
            hr,
        )

        bs = lr.size(0)

        loss_sum += (
            loss.item() * bs
        )
        psnr_sum += (
            psnr_values.sum().item()
        )
        count += bs

    return (
        loss_sum / count,
        psnr_sum / count,
    )


history = {
    "train_loss": [],
    "val_loss": [],
    "val_psnr": [],
    "lr": [],
}

best_val_psnr = -float("inf")
no_improve = 0


for epoch in range(
    1,
    MAX_EPOCH + 1,
):
    sr_train_model.train()

    running_loss = 0.0
    seen = 0

    pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch:02d}/{MAX_EPOCH}",
    )

    for lr, hr, names in pbar:

        lr = lr.to(
            TRAIN_DEVICE,
            non_blocking=True,
        )
        hr = hr.to(
            TRAIN_DEVICE,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        # ★ 바깥 autocast 제거
        # proposed_forward 안에서 RRDBNet만 AMP
        try:
            proposed = proposed_forward(
                sr_train_model,
                lr,
            )
        except RuntimeError as e:
            print("\nForward failed on:", names)
            raise

        # ★ L1은 FP32
        loss = criterion(
            proposed.float(),
            hr.float(),
        )

        if not torch.isfinite(loss):
            print("\nNaN/Inf loss on:", names)
            print(
                "lr range:",
                lr.min().item(),
                lr.max().item(),
            )
            print(
                "hr range:",
                hr.min().item(),
                hr.max().item(),
            )
            print(
                "proposed range:",
                proposed.min().item(),
                proposed.max().item(),
            )
            raise RuntimeError(
                "Training loss became NaN/Inf"
            )

        # ============================================================
        # Backward + AMP overflow 대응 + Gradient clipping
        # ============================================================

        scaler.scale(loss).backward()

        # AMP scaling을 원래 gradient 크기로 복원
        scaler.unscale_(optimizer)


        # ------------------------------------------------------------
        # gradient finite 검사
        # ------------------------------------------------------------

        grad_is_finite = True

        for param in sr_train_model.parameters():

            if param.grad is None:
                continue

            if not torch.isfinite(param.grad).all():
                grad_is_finite = False
                break


        # ------------------------------------------------------------
        # NaN / Inf gradient가 발생한 경우
        #
        # AMP에서는 특정 batch에서 overflow가 발생할 수 있음.
        # 학습 전체를 종료하지 않고 해당 optimizer step만 skip.
        # GradScaler가 scale을 자동 조절하도록 함.
        # ------------------------------------------------------------

        if not grad_is_finite:

            print(
                f"\n[AMP overflow] "
                f"NaN/Inf gradient on {names}"
            )

            print(
                f"loss = {loss.item():.6f}"
            )

            print(
                f"scale = {scaler.get_scale():.1f}"
            )

            # scaler.step()은 inf/nan gradient를 확인하면
            # optimizer.step()을 자동으로 skip함
            scaler.step(
                optimizer
            )

            # scale 감소
            scaler.update()

            optimizer.zero_grad(
                set_to_none=True
            )

            # 다음 batch로 진행
            continue


        # ------------------------------------------------------------
        # 정상 gradient일 때만 clipping
        # ------------------------------------------------------------

        grad_norm = torch.nn.utils.clip_grad_norm_(
            sr_train_model.parameters(),
            max_norm=GRAD_CLIP,
        )


        # ------------------------------------------------------------
        # optimizer update
        # ------------------------------------------------------------

        scaler.step(
            optimizer
        )

        scaler.update()

        bs = lr.size(0)

        running_loss += (
            loss.item() * bs
        )
        seen += bs

        pbar.set_postfix(
            loss=f"{running_loss / seen:.5f}",
            grad=f"{float(grad_norm):.3f}",
            lr=(
                f"{optimizer.param_groups[0]['lr']:.2e}"
            ),
        )

    train_loss = (
        running_loss / seen
    )

    val_loss, val_psnr = (
        evaluate_proposed(
            sr_train_model,
            val_loader,
            desc=f"Val {epoch:02d}",
        )
    )

    # Validation PSNR 기준 scheduler
    scheduler.step(
        val_psnr
    )

    current_lr = (
        optimizer.param_groups[0]["lr"]
    )

    history["train_loss"].append(
        train_loss
    )
    history["val_loss"].append(
        val_loss
    )
    history["val_psnr"].append(
        val_psnr
    )
    history["lr"].append(
        current_lr
    )

    improved = (
        val_psnr
        > best_val_psnr + 1e-6
    )

    if improved:
        best_val_psnr = val_psnr
        no_improve = 0

        torch.save(
            {
                "params_ema": {
                    k: v.detach().cpu()
                    for k, v
                    in sr_train_model
                    .state_dict()
                    .items()
                },
                "epoch": epoch,
                "val_psnr": val_psnr,
                "val_loss": val_loss,
                "config": {
                    "optimizer": "Adam",
                    "lr": LEARNING_RATE,
                    "weight_decay": WEIGHT_DECAY,
                    "loss": (
                        "L1Loss on Proposed "
                        "final output"
                    ),
                    "max_epoch": MAX_EPOCH,
                    "scheduler": (
                        "ReduceLROnPlateau"
                    ),
                    "early_stopping_patience":
                        EARLY_STOP_PATIENCE,
                    "grad_clip": GRAD_CLIP,
                    "amp": AMP_ENABLED,
                    "train_count":
                        len(train_dataset),
                    "val_count":
                        len(val_dataset),
                    "test_count":
                        len(test_dataset),
                },
            },
            BEST_PATH,
        )

        mark = "★ BEST SAVED"

    else:
        no_improve += 1
        mark = (
            f"no improve "
            f"{no_improve}/"
            f"{EARLY_STOP_PATIENCE}"
        )

    print(
        f"[Epoch {epoch:02d}] "
        f"train L1={train_loss:.6f} | "
        f"val L1={val_loss:.6f} | "
        f"val PSNR={val_psnr:.4f} dB | "
        f"lr={current_lr:.2e} | "
        f"{mark}"
    )

    if (
        no_improve
        >= EARLY_STOP_PATIENCE
    ):
        print(
            "\nEarly stopping: "
            f"validation PSNR가 "
            f"{EARLY_STOP_PATIENCE}회 연속 "
            "개선되지 않았습니다."
        )
        break


print("\nTraining finished")
print(
    "Best validation PSNR:",
    best_val_psnr,
)
print(
    "Best checkpoint:",
    BEST_PATH,
)


In [ ]:
# ============================================================
# 6. Best checkpoint -> Test 100장 평가 + 기존 upsampler에 연결
# ============================================================

import matplotlib.pyplot as plt
from realesrgan import RealESRGANer

best_ckpt = torch.load(BEST_PATH, map_location='cpu')
sr_train_model.load_state_dict(best_ckpt['params_ema'], strict=True)
sr_train_model = sr_train_model.to(TRAIN_DEVICE).eval()

# Test는 학습/조기종료에 전혀 사용하지 않고 best model 확정 후 1회 평가
test_loss, test_psnr = evaluate_proposed(
    sr_train_model,
    test_loader,
    desc='Test 100 images',
)

print('==========================================')
print('FINAL DIV2K TEST')
print('==========================================')
print('Train count      :', len(train_dataset))
print('Validation count :', len(val_dataset))
print('Test count       :', len(test_dataset))
print(f'Test L1          : {test_loss:.6f}')
print(f'Test PSNR        : {test_psnr:.4f} dB')
print('Best checkpoint  :', BEST_PATH)
print('==========================================')

# 학습 곡선
fig = plt.figure(figsize=(8, 5))
plt.plot(range(1, len(history['train_loss']) + 1), history['train_loss'], label='Train L1')
plt.plot(range(1, len(history['val_loss']) + 1), history['val_loss'], label='Val L1')
plt.xlabel('Epoch')
plt.ylabel('L1 Loss')
plt.title('Fine-tuning Loss')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

fig = plt.figure(figsize=(8, 5))
plt.plot(range(1, len(history['val_psnr']) + 1), history['val_psnr'], marker='o')
plt.xlabel('Epoch')
plt.ylabel('Validation PSNR (dB)')
plt.title('Validation PSNR')
plt.grid(alpha=0.3)
plt.show()

# ------------------------------------------------------------
# 기존 아래쪽 inference 코드를 최대한 그대로 쓰기 위해
# upsampler만 best fine-tuned checkpoint로 교체
# ------------------------------------------------------------

best_infer_model = RRDBNet(
    num_in_ch=3,
    num_out_ch=3,
    num_feat=64,
    num_block=23,
    num_grow_ch=32,
    scale=4,
)

upsampler = RealESRGANer(
    scale=4,
    model_path=BEST_PATH,
    model=best_infer_model,
    tile=0,
    tile_pad=10,
    pre_pad=0,
    half=torch.cuda.is_available(),
)

print('Fine-tuned best model is now connected to `upsampler`.')
print('아래 기존 Depth + Structure 단일 이미지 실험 셀은 그대로 실행하면 됩니다.')


In [ ]:

# ============================================================
# Baseline vs Proposed
# Train L1 Loss / Validation PSNR 비교
# + Baseline Early Stopping만 표시
# + Validation PSNR 동일 Epoch에 점 표시
# + Smooth Curve
# ============================================================
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator

CHECKPOINT_DIR = '/content/drive/MyDrive/KSY/capstone/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
BEST_PATH = os.path.join(CHECKPOINT_DIR, 'best_proposed_realesrgan_div2k.pth')
# ============================================================
# 1. BASELINE 결과
# ============================================================

baseline_epoch = np.arange(1, 29)

baseline_train_l1 = np.array([
    0.035278,
    0.034051,
    0.032447,
    0.032897,
    0.032044,
    0.032216,
    0.031420,
    0.031962,
    0.030990,
    0.032500,
    0.032318,
    0.032144,
    0.031383,
    0.032128,
    0.032837,
    0.031644,
    0.031343,
    0.031856,
    0.031895,
    0.031803,
    0.031751,
    0.031008,
    0.032169,
    0.031361,
    0.032010,
    0.031625,
    0.031905,
    0.031268,
])

baseline_val_psnr = np.array([
    np.nan,
    26.159586,
    np.nan,
    26.259991,
    np.nan,
    26.367745,
    np.nan,
    26.324706,
    np.nan,
    26.416331,
    np.nan,
    26.435515,
    np.nan,
    26.428675,
    np.nan,
    26.443557,
    np.nan,
    26.422050,
    np.nan,
    26.525065,
    np.nan,
    26.466841,
    np.nan,
    26.527446,
    np.nan,
    26.492032,
    np.nan,
    26.493981,
])


# ============================================================
# 2. PROPOSED 결과
# ============================================================

proposed_train_l1 = np.asarray(
    history["train_loss"],
    dtype=float
)

proposed_val_psnr = np.asarray(
    history["val_psnr"],
    dtype=float
)

proposed_epoch = np.arange(
    1,
    len(proposed_train_l1) + 1
)

assert len(proposed_train_l1) == len(proposed_val_psnr), \
    "Proposed Train Loss와 Val PSNR 길이가 다릅니다."


# ============================================================
# 3. Baseline Early Stopping
# ============================================================

baseline_early_stop_epoch = 28

print("Baseline epochs :", len(baseline_epoch))
print("Proposed epochs :", len(proposed_epoch))
print("Baseline Early Stop Epoch :", baseline_early_stop_epoch)


# ============================================================
# 4. Smooth Curve 함수
# ============================================================

def plot_smooth_curve(
    x,
    y,
    color,
    label,
    marker_size=5,
    linewidth=2.2
):
    x = np.asarray(x)
    y = np.asarray(y)

    mask = np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if len(x) == 0:
        return

    # 측정값이 2개 이상일 때 부드러운 곡선
    if len(x) >= 2:
        x_smooth = np.linspace(
            x.min(),
            x.max(),
            500
        )

        interpolator = PchipInterpolator(x, y)
        y_smooth = interpolator(x_smooth)

        plt.plot(
            x_smooth,
            y_smooth,
            color=color,
            linewidth=linewidth,
            label=label
        )

    # 실제 측정 지점에 점 표시
    plt.plot(
        x,
        y,
        linestyle="None",
        marker="o",
        markersize=marker_size,
        color=color
    )


# ============================================================
# 5. Train L1 Loss 비교
# ============================================================

plt.figure(figsize=(10, 6))

# Baseline
plt.plot(
    baseline_epoch,
    baseline_train_l1,
    marker="o",
    linewidth=2,
    markersize=4,
    color="tab:blue",
    label="Baseline"
)

# Proposed
plt.plot(
    proposed_epoch,
    proposed_train_l1,
    marker="o",
    linewidth=2,
    markersize=4,
    color="tab:red",
    label="Proposed"
)


# ------------------------------------------------------------
# Baseline Early Stopping만 표시
# ------------------------------------------------------------

plt.axvline(
    x=baseline_early_stop_epoch,
    color="tab:blue",
    linestyle="--",
    linewidth=1.8,
    alpha=0.8,
    label=f"Baseline Early Stop (Epoch {baseline_early_stop_epoch})"
)

plt.scatter(
    baseline_early_stop_epoch,
    baseline_train_l1[-1],
    s=200,
    marker="*",
    color="tab:blue",
    edgecolor="black",
    zorder=5
)

plt.annotate(
    f"Baseline Stop\nEpoch {baseline_early_stop_epoch}",
    xy=(
        baseline_early_stop_epoch,
        baseline_train_l1[-1]
    ),
    xytext=(
        baseline_early_stop_epoch - 7,
        baseline_train_l1[-1] + 0.002
    ),
    arrowprops=dict(
        arrowstyle="->",
        color="tab:blue"
    ),
    color="tab:blue",
    fontsize=10
)


# ------------------------------------------------------------
# Graph Style
# ------------------------------------------------------------

plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Train L1 Loss", fontsize=12)

plt.title(
    "Baseline vs Proposed - Training L1 Loss",
    fontsize=14
)

plt.grid(True, alpha=0.3)
plt.legend(fontsize=10)

plt.xlim(
    1,
    max(baseline_epoch[-1], proposed_epoch[-1]) + 1
)

plt.tight_layout()
plt.show()


# ============================================================
# 6. Validation PSNR 비교
# ============================================================

# Baseline 실제 PSNR 측정 Epoch
baseline_psnr_mask = np.isfinite(baseline_val_psnr)

baseline_psnr_epochs = baseline_epoch[
    baseline_psnr_mask
]

baseline_psnr_values = baseline_val_psnr[
    baseline_psnr_mask
]


# Proposed: 짝수 Epoch에 점 표시 (2, 4, ..., 30)
proposed_psnr_mask = (
    np.isfinite(proposed_val_psnr)
    & (proposed_epoch % 2 == 0)
)

proposed_psnr_epochs = proposed_epoch[proposed_psnr_mask]
proposed_psnr_values = proposed_val_psnr[proposed_psnr_mask]
proposed_psnr_epochs = proposed_epoch[
    proposed_psnr_mask
]

proposed_psnr_values = proposed_val_psnr[
    proposed_psnr_mask
]


plt.figure(figsize=(10, 6))


# ------------------------------------------------------------
# Baseline PSNR - Smooth Curve
# ------------------------------------------------------------

plot_smooth_curve(
    baseline_psnr_epochs,
    baseline_psnr_values,
    color="tab:blue",
    label="Baseline",
    marker_size=5
)


# ------------------------------------------------------------
# Proposed PSNR - Smooth Curve
# ------------------------------------------------------------

plot_smooth_curve(
    proposed_psnr_epochs,
    proposed_psnr_values,
    color="tab:red",
    label="Proposed",
    marker_size=5
)


# ------------------------------------------------------------
# Baseline Early Stopping만 표시
# ------------------------------------------------------------

plt.axvline(
    x=baseline_early_stop_epoch,
    color="tab:blue",
    linestyle="--",
    linewidth=1.8,
    alpha=0.8,
    label=f"Baseline Early Stop (Epoch {baseline_early_stop_epoch})"
)

baseline_last_valid_idx = np.where(
    baseline_psnr_mask
)[0][-1]

baseline_last_psnr_epoch = baseline_epoch[
    baseline_last_valid_idx
]

baseline_last_psnr = baseline_val_psnr[
    baseline_last_valid_idx
]

plt.scatter(
    baseline_last_psnr_epoch,
    baseline_last_psnr,
    s=200,
    marker="*",
    color="tab:blue",
    edgecolor="black",
    zorder=5
)

plt.annotate(
    f"Baseline Stop\nEpoch {baseline_early_stop_epoch}",
    xy=(
        baseline_last_psnr_epoch,
        baseline_last_psnr
    ),
    xytext=(
        baseline_last_psnr_epoch - 7,
        baseline_last_psnr - 0.15
    ),
    arrowprops=dict(
        arrowstyle="->",
        color="tab:blue"
    ),
    color="tab:blue",
    fontsize=10
)


# ------------------------------------------------------------
# Graph Style
# ------------------------------------------------------------

plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Validation PSNR (dB)", fontsize=12)

plt.title(
    "Baseline vs Proposed - Validation PSNR",
    fontsize=14
)

plt.grid(True, alpha=0.3)
plt.legend(fontsize=10)

plt.xlim(
    1,
    max(baseline_early_stop_epoch, proposed_epoch[-1]) + 2
)

plt.tight_layout()
plt.show()


# ============================================================
# 7. Best Validation PSNR 비교
# ============================================================

baseline_best_idx = np.nanargmax(baseline_val_psnr)
proposed_best_idx = np.nanargmax(proposed_val_psnr)

baseline_best_psnr = baseline_val_psnr[baseline_best_idx]
proposed_best_psnr = proposed_val_psnr[proposed_best_idx]

print("\n" + "=" * 55)
print("BEST VALIDATION PSNR COMPARISON")
print("=" * 55)

print(
    f"Baseline Best PSNR : {baseline_best_psnr:.4f} dB "
    f"(Epoch {baseline_epoch[baseline_best_idx]})"
)

print(
    f"Proposed Best PSNR : {proposed_best_psnr:.4f} dB "
    f"(Epoch {proposed_epoch[proposed_best_idx]})"
)

print(
    f"PSNR Improvement   : "
    f"{proposed_best_psnr - baseline_best_psnr:+.4f} dB"
)

print("=" * 55)


In [ ]:
import torch

BEST_PATH = "/content/drive/MyDrive/KSY/capstone/checkpoints/best_proposed_realesrgan_div2k.pth"

ckpt = torch.load(
    BEST_PATH,
    map_location="cpu",
    weights_only=True
)

print("Checkpoint keys:")
print(list(ckpt.keys()))

print("\nEpoch:", ckpt.get("epoch", "저장되지 않음"))
print("Val PSNR:", ckpt.get("val_psnr", "저장되지 않음"))

print("\nHistory:", ckpt.get("history", "저장되지 않음"))

In [ ]:
# ============================================================
# THREE-WAY COMPARISON
# 1) Original pretrained Real-ESRGAN x4plus
# 2) Best fine-tuned RRDBNet model only
# 3) Same best model + training-time Depth/Structure idea
# ============================================================
# ============================================================
# BEST CHECKPOINT PROPOSED - ONE-CELL INFERENCE
# Training pipeline matched to original DIV2K notebook
# ============================================================
# Requirements: torch, torchvision, transformers, basicsr, cv2,
# numpy, matplotlib, Pillow, and their compatible dependencies.
# In Colab, mount Google Drive beforehand.

import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torchvision.transforms.functional as TVF
from PIL import Image
from basicsr.archs.rrdbnet_arch import RRDBNet
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

IMAGE_PATH = "/content/anime3.png"
SAVE_PATH = "/content/drive/MyDrive/KSY/capstone/results/proposed"
BEST_PATH = "/content/drive/MyDrive/KSY/capstone/checkpoints/best_proposed_realesrgan_div2k.pth"
ORIGINAL_PATH = "/content/Real-ESRGAN/weights/realesrgan-x4plus.pth"
MAKE_LOW_RESOLUTION = True
DOWNSCALE = 4
UPSCALE = 4
SCALE = 4

# Exactly the training-time Proposed parameters
DEPTH_FAR = 0.10
DEPTH_NEAR = 0.30
EDGE_WEIGHT = 0.4
RESIDUAL_WEIGHT = 0.6
RESIDUAL_PERCENTILE = 80
STRUCTURE_START = 0.15
STRUCTURE_PROTECT_POWER = 0.4
MIN_STRUCTURE_BLUR = 0.03
BLUR_SIGMA = 2.5
MASK_SIGMA = 1.0  # trained setting, NOT the older experiment's 0.5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TRAIN_DEVICE = device
AMP_ENABLED = False  # inference in FP32 for numerical stability

assert os.path.isfile(BEST_PATH), f"Checkpoint not found: {BEST_PATH}"
assert os.path.isfile(IMAGE_PATH), f"Input image not found: {IMAGE_PATH}"

# Original pretrained RRDBNet. Do not accidentally reuse fine-tuned weights.
assert os.path.isfile(ORIGINAL_PATH), (
    f"Original Real-ESRGAN weights missing: {ORIGINAL_PATH}\n"
    "Place the official RealESRGAN_x4plus.pth file there first."
)
original_model = RRDBNet(
    num_in_ch=3, num_out_ch=3,
    num_feat=64, num_block=23, num_grow_ch=32, scale=4,
).to(device)
original_ckpt = torch.load(ORIGINAL_PATH, map_location="cpu", weights_only=True)
original_state = original_ckpt.get("params_ema", original_ckpt.get("params", original_ckpt))
original_model.load_state_dict(original_state, strict=True)
original_model.eval()
print("Loaded ORIGINAL Real-ESRGAN:", ORIGINAL_PATH)

# Depth Anything V2 Small: same as training notebook
MODEL_ID = "depth-anything/Depth-Anything-V2-Small-hf"
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
depth_model = AutoModelForDepthEstimation.from_pretrained(MODEL_ID).to(device).eval()
for p in depth_model.parameters():
    p.requires_grad_(False)

# RRDBNet architecture must match training checkpoint
sr_train_model = RRDBNet(
    num_in_ch=3, num_out_ch=3,
    num_feat=64, num_block=23, num_grow_ch=32, scale=4,
).to(device)
ckpt = torch.load(BEST_PATH, map_location="cpu", weights_only=True)
if not isinstance(ckpt, dict) or "params_ema" not in ckpt:
    raise KeyError("Training checkpoint must contain 'params_ema'")
sr_train_model.load_state_dict(ckpt["params_ema"], strict=True)
sr_train_model.eval()
print("Loaded BEST Proposed checkpoint:", BEST_PATH)
print("Checkpoint epoch:", ckpt.get("epoch", "unknown"))
print("Checkpoint val PSNR:", ckpt.get("val_psnr", "unknown"))

def _odd_kernel_from_sigma(sigma, minimum=3):
    k = max(minimum, int(2 * round(3 * sigma) + 1))
    return k if k % 2 == 1 else k + 1


def gaussian_blur_tensor(x, sigma):
    if sigma <= 0:
        return x
    k = _odd_kernel_from_sigma(sigma)
    return TVF.gaussian_blur(x, [k, k], [sigma, sigma])


def _check_finite(name, x):
    """NaN/Inf가 처음 생기는 지점을 정확히 표시."""
    if not torch.isfinite(x).all():
        finite = x[torch.isfinite(x)]
        if finite.numel() > 0:
            mn = finite.min().item()
            mx = finite.max().item()
        else:
            mn = float("nan")
            mx = float("nan")
        n_nan = torch.isnan(x).sum().item()
        n_inf = torch.isinf(x).sum().item()
        raise RuntimeError(
            f"[{name}] NaN/Inf detected | "
            f"nan={n_nan}, inf={n_inf}, finite_min={mn}, finite_max={mx}"
        )


def depth_map_from_lr(lr, out_size):
    """
    Depth Anything V2는 conditioning map 생성 전용.
    바깥 autocast와 무관하게 반드시 FP32로 실행.
    반환: [B,1,H,W], 0=far / 1=near
    """
    pil_images = []
    lr_cpu = lr.detach().float().cpu().clamp(0.0, 1.0)

    for img in lr_cpu:
        arr = (
            img.permute(1, 2, 0).numpy() * 255.0
        ).round().astype(np.uint8)
        # Pillow 13 대비: mode='RGB' 제거
        pil_images.append(Image.fromarray(arr))

    inputs = processor(images=pil_images, return_tensors="pt")

    # floating tensor는 FP32, 그 외 tensor는 dtype 유지
    inputs = {
        k: (
            v.to(device=device, dtype=torch.float32)
            if torch.is_floating_point(v)
            else v.to(device)
        )
        for k, v in inputs.items()
    }

    with torch.no_grad():
        # ★ 외부 AMP가 켜져 있어도 Depth Anything은 FP32
        with torch.amp.autocast(device_type="cuda", enabled=False):
            pred = depth_model(**inputs).predicted_depth
            pred = pred.unsqueeze(1).float()
            pred = F.interpolate(
                pred,
                size=out_size,
                mode="bicubic",
                align_corners=False,
            )

            _check_finite("depth_raw", pred)

            flat = pred.flatten(1)
            low = torch.quantile(
                flat, 0.02, dim=1
            ).view(-1, 1, 1, 1)
            high = torch.quantile(
                flat, 0.98, dim=1
            ).view(-1, 1, 1, 1)

            # 1e-8 대신 FP32에서 충분히 안전한 하한
            denom = (high - low).clamp_min(1e-6)

            depth = (
                (pred - low) / denom
            ).clamp(0.0, 1.0)

            _check_finite("depth_norm", depth)

    return depth


def make_edge_structure_map_torch(img_rgb, blur_sigma=1.5):
    img_rgb = img_rgb.float()

    gray = (
        0.2989 * img_rgb[:, 0:1]
        + 0.5870 * img_rgb[:, 1:2]
        + 0.1140 * img_rgb[:, 2:3]
    )

    kx = torch.tensor(
        [
            [-1., 0., 1.],
            [-2., 0., 2.],
            [-1., 0., 1.],
        ],
        device=img_rgb.device,
        dtype=torch.float32,
    ).view(1, 1, 3, 3)

    ky = kx.transpose(2, 3)

    gx = F.conv2d(gray, kx, padding=1)
    gy = F.conv2d(gray, ky, padding=1)

    mag = torch.sqrt(
        gx.square() + gy.square() + 1e-12
    )

    norm = torch.quantile(
        mag.detach().flatten(1),
        0.99,
        dim=1,
    ).view(-1, 1, 1, 1)

    norm = norm.clamp_min(1e-6)

    mag = (
        mag / norm
    ).clamp(0.0, 1.0)

    out = gaussian_blur_tensor(
        mag, blur_sigma
    ).clamp(0.0, 1.0)

    _check_finite("edge_map", out)
    return out


def make_residual_structure_map_torch(
    I_up,
    I_sr,
    percentile=80,
    blur_sigma=2.0,
):
    I_up = I_up.float()
    I_sr = I_sr.float()

    diff = torch.abs(
        I_sr - I_up
    ).mean(dim=1, keepdim=True)

    _check_finite("residual_diff", diff)

    q = percentile / 100.0
    flat = diff.detach().flatten(1)

    threshold = torch.quantile(
        flat, q, dim=1
    ).view(-1, 1, 1, 1)

    high = torch.quantile(
        flat, 0.99, dim=1
    ).view(-1, 1, 1, 1)

    denom = (
        high - threshold
    ).clamp_min(1e-6)

    residual = (
        (diff - threshold) / denom
    ).clamp(0.0, 1.0)

    residual = gaussian_blur_tensor(
        residual, blur_sigma
    ).clamp(0.0, 1.0)

    _check_finite("residual_map", residual)
    return residual


def proposed_forward(sr_model, lr, return_maps=False):
    """
    RRDBNet convolution은 AMP 사용.
    Depth / quantile / structure / blur / fusion은 FP32.
    최종 I_final에서 RRDBNet까지 gradient가 흐름.
    """

    # --------------------------------------------------------
    # 1. RRDBNet만 AMP
    # --------------------------------------------------------
    with torch.amp.autocast(
        device_type="cuda",
        enabled=AMP_ENABLED,
    ):
        I_sr = sr_model(lr)

    _check_finite("rrdbnet_output", I_sr)

    # 여기부터 FP32
    I_sr_fp32 = I_sr.float()

    I_up = F.interpolate(
        lr.float(),
        scale_factor=SCALE,
        mode="bicubic",
        align_corners=False,
    )

    _check_finite("bicubic_up", I_up)

    # --------------------------------------------------------
    # 2. Depth
    # --------------------------------------------------------
    M_depth = depth_map_from_lr(
        lr,
        I_sr.shape[-2:],
    ).to(
        device=I_sr.device,
        dtype=torch.float32,
    )

    # --------------------------------------------------------
    # 3. Edge
    # --------------------------------------------------------
    M_edge = make_edge_structure_map_torch(
        I_up,
        blur_sigma=1.5,
    )

    # --------------------------------------------------------
    # 4. Residual
    # --------------------------------------------------------
    M_residual = make_residual_structure_map_torch(
        I_up,
        I_sr_fp32,
        percentile=RESIDUAL_PERCENTILE,
        blur_sigma=2.0,
    )

    S = (
        EDGE_WEIGHT * M_edge
        + RESIDUAL_WEIGHT * M_residual
    ).clamp(0.0, 1.0)

    _check_finite("structure_map", S)

    # --------------------------------------------------------
    # 5. Depth -> far
    # --------------------------------------------------------
    D = M_depth.clamp(0.0, 1.0)

    far_strength = 1.0 - (
        (D - DEPTH_FAR)
        / max(DEPTH_NEAR - DEPTH_FAR, 1e-6)
    ).clamp(0.0, 1.0)

    far_strength = (
        far_strength
        * far_strength
        * (3.0 - 2.0 * far_strength)
    )

    _check_finite("far_strength", far_strength)

    # --------------------------------------------------------
    # 6. Structure protection
    # --------------------------------------------------------
    structure_raw = (
    (S - STRUCTURE_START)
    / max(1.0 - STRUCTURE_START, 1e-6)
    ).clamp(0.0, 1.0)

    POWER_EPS = 1e-4

    eps_pow = POWER_EPS ** STRUCTURE_PROTECT_POWER

    structure_protect = (
        (structure_raw + POWER_EPS).pow(
            STRUCTURE_PROTECT_POWER
        )
        - eps_pow
    )

    structure_protect = structure_protect / (
        (1.0 + POWER_EPS) ** STRUCTURE_PROTECT_POWER
        - eps_pow
    )

    structure_protect = structure_protect.clamp(
        0.0,
        1.0
    )

    _check_finite(
        "structure_protect",
        structure_protect
    )

    _check_finite(
        "structure_protect",
        structure_protect,
    )

    # --------------------------------------------------------
    # 7. Blur strength
    # --------------------------------------------------------
    blur_strength = far_strength * (
        1.0
        - (1.0 - MIN_STRUCTURE_BLUR)
        * structure_protect
    )

    blur_strength = gaussian_blur_tensor(
        blur_strength,
        MASK_SIGMA,
    ).clamp(0.0, 1.0)

    _check_finite("blur_strength", blur_strength)

    # --------------------------------------------------------
    # 8. Proposed final
    # --------------------------------------------------------
    I_blur = gaussian_blur_tensor(
        I_sr_fp32,
        BLUR_SIGMA,
    )

    _check_finite("blurred_sr", I_blur)

    I_final = (
        (1.0 - blur_strength) * I_sr_fp32
        + blur_strength * I_blur
    )

    _check_finite("I_final", I_final)

    if return_maps:
        maps = {
            "I_sr": I_sr_fp32,
            "I_up": I_up,
            "depth": D,
            "edge": M_edge,
            "residual": M_residual,
            "structure": S,
            "far_strength": far_strength,
            "structure_protect": structure_protect,
            "blur_strength": blur_strength,
        }
        return I_final, maps

    return I_final



# ============================================================
# Image -> EXACT training-time proposed_forward(), save visuals
# ============================================================
image_name = os.path.splitext(os.path.basename(IMAGE_PATH))[0]
experiment_dir = os.path.join(SAVE_PATH, image_name)
original_dir = os.path.join(experiment_dir, "00_original_realesrgan")
model_only_dir = os.path.join(experiment_dir, "01_model_only")
idea_dir = os.path.join(experiment_dir, "02_idea_applied")
for directory in (experiment_dir, original_dir, model_only_dir, idea_dir):
    os.makedirs(directory, exist_ok=True)

hr_bgr = cv2.imread(IMAGE_PATH, cv2.IMREAD_COLOR)
if hr_bgr is None:
    raise FileNotFoundError(IMAGE_PATH)
hr_h, hr_w = hr_bgr.shape[:2]
if MAKE_LOW_RESOLUTION:
    # Match DIV2K training pipeline: bicubic x4 downsampling in RGB PIL
    lr_pil = Image.fromarray(cv2.cvtColor(hr_bgr, cv2.COLOR_BGR2RGB))
    lr_pil = lr_pil.resize((max(1, hr_w // DOWNSCALE), max(1, hr_h // DOWNSCALE)), Image.Resampling.BICUBIC)
    lr_rgb = np.asarray(lr_pil).copy()
    lr_bgr = cv2.cvtColor(lr_rgb, cv2.COLOR_RGB2BGR)
else:
    lr_bgr = hr_bgr.copy()
    lr_rgb = cv2.cvtColor(lr_bgr, cv2.COLOR_BGR2RGB)

h, w = lr_bgr.shape[:2]
lr_tensor = torch.from_numpy(lr_rgb.copy()).permute(2, 0, 1).unsqueeze(0).float().div(255.0).to(device)

with torch.inference_mode():
    original_sr_tensor = original_model(lr_tensor).float()
    final, maps = proposed_forward(sr_train_model, lr_tensor, return_maps=True)


def rgb_tensor_to_bgr(x):
    arr = (x[0].detach().float().clamp(0, 1).permute(1, 2, 0).cpu().numpy() * 255.0).round().astype(np.uint8)
    return cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)


def map_to_gray(x):
    z = x[0, 0].detach().float().clamp(0, 1).cpu().numpy()
    return (z * 255.0).round().astype(np.uint8)


def save_png(filename, img, directory=None):
    path = os.path.join(directory or experiment_dir, filename)
    if not cv2.imwrite(path, img):
        raise RuntimeError(f"Could not save {path}")
    return path

I_original = rgb_tensor_to_bgr(original_sr_tensor)
I_sr = rgb_tensor_to_bgr(maps["I_sr"])
I_final = rgb_tensor_to_bgr(final)
I_up = cv2.resize(lr_bgr, (I_sr.shape[1], I_sr.shape[0]), interpolation=cv2.INTER_LANCZOS4)
# Lanczos is for display only. Training used bicubic I_up for its maps.

save_png("01_lr_input.png", lr_bgr)
save_png("03_original_realesrgan.png", I_original, original_dir)
save_png("03_original_realesrgan.png", I_original)
save_png("02_lanczos.png", I_up)
# ① BEST 체크포인트의 원본 출력: 추가 blur/fusion 없음
save_png("model_only_no_blur.png", I_sr, model_only_dir)
save_png("03_realesrgan.png", I_sr)  # 기존 파일명 호환
save_png("04_depth_gray.png", map_to_gray(maps["depth"]))
save_png("06_edge_map.png", map_to_gray(maps["edge"]))
save_png("07_residual_map.png", map_to_gray(maps["residual"]))
save_png("08_structure_map.png", map_to_gray(maps["structure"]))
save_png("09_far_strength.png", map_to_gray(maps["far_strength"]))
save_png("10_structure_protection.png", map_to_gray(maps["structure_protect"]))
save_png("11_blur_weight.png", map_to_gray(maps["blur_strength"]))
with torch.inference_mode():
    blurred = gaussian_blur_tensor(maps["I_sr"], BLUR_SIGMA)
save_png("12_realesrgan_blurred.png", rgb_tensor_to_bgr(blurred))
# ② 동일 체크포인트 출력에 학습 당시의 Depth/Structure/Blur 아이디어 적용
save_png("idea_applied.png", I_final, idea_dir)
save_png("13_proposed.png", I_final)  # 기존 파일명 호환

diff = np.abs(I_sr.astype(np.float32) - I_final.astype(np.float32)).mean(axis=2)
np.save(os.path.join(experiment_dir, "14_difference_raw.npy"), diff)
scale = max(float(np.percentile(diff, 99)), 1e-8)
save_png("15_difference_gray.png", (np.clip(diff / scale, 0, 1) * 255).astype(np.uint8))

D = maps["depth"][0,0].cpu().numpy()
S = maps["structure"][0,0].cpu().numpy()
far = maps["far_strength"][0,0].cpu().numpy()
prot = maps["structure_protect"][0,0].cpu().numpy()
blur_weight = maps["blur_strength"][0,0].cpu().numpy()

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(D, cmap="inferno", vmin=0, vmax=1)
ax.set_title("Depth Anything V2 (0=Far, 1=Near)")
ax.axis("off")
fig.colorbar(im, ax=ax)
fig.tight_layout()
fig.savefig(os.path.join(experiment_dir, "05_depth_visualization.png"), dpi=200, bbox_inches="tight")
plt.close(fig)

fig, axes = plt.subplots(2, 4, figsize=(22, 11))
items = [
    (D, "Depth", "inferno"),
    (far, "Far Strength", "gray"),
    (S, "Structure", "gray"),
    (prot, "Structure Protection", "viridis"),
    (cv2.cvtColor(lr_bgr, cv2.COLOR_BGR2RGB), "LR Input", None),
    (cv2.cvtColor(I_sr, cv2.COLOR_BGR2RGB), "Fine-tuned RRDBNet", None),
    (blur_weight, "Blur Weight", "magma"),
    (cv2.cvtColor(I_final, cv2.COLOR_BGR2RGB), "Proposed (Best Checkpoint)", None),
]
for ax, (arr, title, cmap) in zip(axes.flat, items):
    ax.imshow(arr, cmap=cmap, vmin=0 if cmap else None, vmax=1 if cmap else None)
    ax.set_title(title)
    ax.axis("off")
fig.tight_layout()
fig.savefig(os.path.join(experiment_dir, "16_internal_visualization.png"), dpi=200, bbox_inches="tight")
plt.close(fig)

# Three-way comparison: all outputs from the EXACT same LR input
fig, axes = plt.subplots(1, 4, figsize=(24, 7))
comparison = [
    (lr_bgr, "Input LR"),
    (I_original, "1. Original Real-ESRGAN"),
    (I_sr, "2. Best Model Only"),
    (I_final, "3. Best Model + Idea"),
]
for ax, (im, title) in zip(axes, comparison):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=13)
    ax.axis("off")
fig.tight_layout()
fig.savefig(os.path.join(experiment_dir, "17_three_way_comparison.png"), dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, axes = plt.subplots(1, 4, figsize=(24, 6))
for ax, (im, title) in zip(axes[:3], comparison[1:]):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis("off")
plot = axes[3].imshow(diff, cmap="hot")
axes[3].set_title("Difference: Model Only vs Idea")
axes[3].axis("off")
fig.colorbar(plot, ax=axes[3], fraction=.046, pad=.04)
fig.tight_layout()
fig.savefig(os.path.join(experiment_dir, "18_three_results_and_difference.png"), dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

far_region = far >= 0.5
structured_region = prot >= 0.5
stats = {
    "IMAGE_PATH": IMAGE_PATH,
    "BEST_PATH": BEST_PATH,
    "ORIGINAL_PATH": ORIGINAL_PATH,
    "BEST_EPOCH": ckpt.get("epoch", "unknown"),
    "BEST_VAL_PSNR": ckpt.get("val_psnr", "unknown"),
    "LR_SIZE": f"{w}x{h}",
    "SR_SIZE": f"{I_sr.shape[1]}x{I_sr.shape[0]}",
    "DEPTH_FAR": DEPTH_FAR,
    "DEPTH_NEAR": DEPTH_NEAR,
    "STRUCTURE_START": STRUCTURE_START,
    "STRUCTURE_PROTECT_POWER": STRUCTURE_PROTECT_POWER,
    "MIN_STRUCTURE_BLUR": MIN_STRUCTURE_BLUR,
    "BLUR_SIGMA": BLUR_SIGMA,
    "MASK_SIGMA": MASK_SIGMA,
    "depth_mean": float(D.mean()),
    "structure_mean": float(S.mean()),
    "far_strength_mean": float(far.mean()),
    "blur_weight_mean": float(blur_weight.mean()),
    "difference_mean": float(diff.mean()),
    "far_structure_blur_mean": float(blur_weight[far_region & structured_region].mean()) if np.any(far_region & structured_region) else "N/A",
    "far_no_structure_blur_mean": float(blur_weight[far_region & ~structured_region].mean()) if np.any(far_region & ~structured_region) else "N/A",
}
with open(os.path.join(experiment_dir, "19_parameters_and_stats.txt"), "w", encoding="utf-8") as f:
    for k, v in stats.items(): f.write(f"{k}={v}\n")

print("EXPERIMENT COMPLETE")
print("Loaded checkpoint:", BEST_PATH)
print("Saved to:", experiment_dir)
print("Original Real-ESRGAN:", os.path.join(original_dir, "03_original_realesrgan.png"))
print("Model only (NO additional blur):", os.path.join(model_only_dir, "model_only_no_blur.png"))
print("Idea applied (Depth + Structure + Blur):", os.path.join(idea_dir, "idea_applied.png"))
print("Three-way comparison:", os.path.join(experiment_dir, "17_three_way_comparison.png"))
print("Proposed output:", os.path.join(experiment_dir, "13_proposed.png"))
print("Difference comparison:", os.path.join(experiment_dir, "18_three_results_and_difference.png"))


In [ ]:
# ============================================================
# THREE-WAY COMPARISON
# 1) Original pretrained Real-ESRGAN x4plus
# 2) Best fine-tuned RRDBNet model only
# 3) Same best model + training-time Depth/Structure idea
# ============================================================
# ============================================================
# BEST CHECKPOINT PROPOSED - ONE-CELL INFERENCE
# Training pipeline matched to original DIV2K notebook
# ============================================================
# Requirements: torch, torchvision, transformers, basicsr, cv2,
# numpy, matplotlib, Pillow, and their compatible dependencies.
# In Colab, mount Google Drive beforehand.

import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torchvision.transforms.functional as TVF
from PIL import Image
from basicsr.archs.rrdbnet_arch import RRDBNet
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

IMAGE_PATH = "/content/anime3.png"
SAVE_PATH = "/content/drive/MyDrive/KSY/capstone/results/proposed"
BEST_PATH = "/content/drive/MyDrive/KSY/capstone/checkpoints/best_proposed_realesrgan_div2k.pth"
ORIGINAL_PATH = "/content/Real-ESRGAN/weights/realesrgan-x4plus.pth"
MAKE_LOW_RESOLUTION = True
DOWNSCALE = 4
UPSCALE = 4
SCALE = 4

# Exactly the training-time Proposed parameters
DEPTH_FAR = 0.10
DEPTH_NEAR = 0.30
EDGE_WEIGHT = 0.4
RESIDUAL_WEIGHT = 0.6
RESIDUAL_PERCENTILE = 80
STRUCTURE_START = 0.15
STRUCTURE_PROTECT_POWER = 0.4
MIN_STRUCTURE_BLUR = 0.03
BLUR_SIGMA = 2.5
MASK_SIGMA = 1.0  # trained setting, NOT the older experiment's 0.5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TRAIN_DEVICE = device
AMP_ENABLED = False  # inference in FP32 for numerical stability

assert os.path.isfile(BEST_PATH), f"Checkpoint not found: {BEST_PATH}"
assert os.path.isfile(IMAGE_PATH), f"Input image not found: {IMAGE_PATH}"

# Original pretrained RRDBNet. Do not accidentally reuse fine-tuned weights.
assert os.path.isfile(ORIGINAL_PATH), (
    f"Original Real-ESRGAN weights missing: {ORIGINAL_PATH}\n"
    "Place the official RealESRGAN_x4plus.pth file there first."
)
original_model = RRDBNet(
    num_in_ch=3, num_out_ch=3,
    num_feat=64, num_block=23, num_grow_ch=32, scale=4,
).to(device)
original_ckpt = torch.load(ORIGINAL_PATH, map_location="cpu", weights_only=True)
original_state = original_ckpt.get("params_ema", original_ckpt.get("params", original_ckpt))
original_model.load_state_dict(original_state, strict=True)
original_model.eval()
print("Loaded ORIGINAL Real-ESRGAN:", ORIGINAL_PATH)

# Depth Anything V2 Small: same as training notebook
MODEL_ID = "depth-anything/Depth-Anything-V2-Small-hf"
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
depth_model = AutoModelForDepthEstimation.from_pretrained(MODEL_ID).to(device).eval()
for p in depth_model.parameters():
    p.requires_grad_(False)

# RRDBNet architecture must match training checkpoint
sr_train_model = RRDBNet(
    num_in_ch=3, num_out_ch=3,
    num_feat=64, num_block=23, num_grow_ch=32, scale=4,
).to(device)
ckpt = torch.load(BEST_PATH, map_location="cpu", weights_only=True)
if not isinstance(ckpt, dict) or "params_ema" not in ckpt:
    raise KeyError("Training checkpoint must contain 'params_ema'")
sr_train_model.load_state_dict(ckpt["params_ema"], strict=True)
sr_train_model.eval()
print("Loaded BEST Proposed checkpoint:", BEST_PATH)
print("Checkpoint epoch:", ckpt.get("epoch", "unknown"))
print("Checkpoint val PSNR:", ckpt.get("val_psnr", "unknown"))

def _odd_kernel_from_sigma(sigma, minimum=3):
    k = max(minimum, int(2 * round(3 * sigma) + 1))
    return k if k % 2 == 1 else k + 1


def gaussian_blur_tensor(x, sigma):
    if sigma <= 0:
        return x
    k = _odd_kernel_from_sigma(sigma)
    return TVF.gaussian_blur(x, [k, k], [sigma, sigma])


def _check_finite(name, x):
    """NaN/Inf가 처음 생기는 지점을 정확히 표시."""
    if not torch.isfinite(x).all():
        finite = x[torch.isfinite(x)]
        if finite.numel() > 0:
            mn = finite.min().item()
            mx = finite.max().item()
        else:
            mn = float("nan")
            mx = float("nan")
        n_nan = torch.isnan(x).sum().item()
        n_inf = torch.isinf(x).sum().item()
        raise RuntimeError(
            f"[{name}] NaN/Inf detected | "
            f"nan={n_nan}, inf={n_inf}, finite_min={mn}, finite_max={mx}"
        )


def depth_map_from_lr(lr, out_size):
    """
    Depth Anything V2는 conditioning map 생성 전용.
    바깥 autocast와 무관하게 반드시 FP32로 실행.
    반환: [B,1,H,W], 0=far / 1=near
    """
    pil_images = []
    lr_cpu = lr.detach().float().cpu().clamp(0.0, 1.0)

    for img in lr_cpu:
        arr = (
            img.permute(1, 2, 0).numpy() * 255.0
        ).round().astype(np.uint8)
        # Pillow 13 대비: mode='RGB' 제거
        pil_images.append(Image.fromarray(arr))

    inputs = processor(images=pil_images, return_tensors="pt")

    # floating tensor는 FP32, 그 외 tensor는 dtype 유지
    inputs = {
        k: (
            v.to(device=device, dtype=torch.float32)
            if torch.is_floating_point(v)
            else v.to(device)
        )
        for k, v in inputs.items()
    }

    with torch.no_grad():
        # ★ 외부 AMP가 켜져 있어도 Depth Anything은 FP32
        with torch.amp.autocast(device_type="cuda", enabled=False):
            pred = depth_model(**inputs).predicted_depth
            pred = pred.unsqueeze(1).float()
            pred = F.interpolate(
                pred,
                size=out_size,
                mode="bicubic",
                align_corners=False,
            )

            _check_finite("depth_raw", pred)

            flat = pred.flatten(1)
            low = torch.quantile(
                flat, 0.02, dim=1
            ).view(-1, 1, 1, 1)
            high = torch.quantile(
                flat, 0.98, dim=1
            ).view(-1, 1, 1, 1)

            # 1e-8 대신 FP32에서 충분히 안전한 하한
            denom = (high - low).clamp_min(1e-6)

            depth = (
                (pred - low) / denom
            ).clamp(0.0, 1.0)

            _check_finite("depth_norm", depth)

    return depth


def make_edge_structure_map_torch(img_rgb, blur_sigma=1.5):
    img_rgb = img_rgb.float()

    gray = (
        0.2989 * img_rgb[:, 0:1]
        + 0.5870 * img_rgb[:, 1:2]
        + 0.1140 * img_rgb[:, 2:3]
    )

    kx = torch.tensor(
        [
            [-1., 0., 1.],
            [-2., 0., 2.],
            [-1., 0., 1.],
        ],
        device=img_rgb.device,
        dtype=torch.float32,
    ).view(1, 1, 3, 3)

    ky = kx.transpose(2, 3)

    gx = F.conv2d(gray, kx, padding=1)
    gy = F.conv2d(gray, ky, padding=1)

    mag = torch.sqrt(
        gx.square() + gy.square() + 1e-12
    )

    norm = torch.quantile(
        mag.detach().flatten(1),
        0.99,
        dim=1,
    ).view(-1, 1, 1, 1)

    norm = norm.clamp_min(1e-6)

    mag = (
        mag / norm
    ).clamp(0.0, 1.0)

    out = gaussian_blur_tensor(
        mag, blur_sigma
    ).clamp(0.0, 1.0)

    _check_finite("edge_map", out)
    return out


def make_residual_structure_map_torch(
    I_up,
    I_sr,
    percentile=80,
    blur_sigma=2.0,
):
    I_up = I_up.float()
    I_sr = I_sr.float()

    diff = torch.abs(
        I_sr - I_up
    ).mean(dim=1, keepdim=True)

    _check_finite("residual_diff", diff)

    q = percentile / 100.0
    flat = diff.detach().flatten(1)

    threshold = torch.quantile(
        flat, q, dim=1
    ).view(-1, 1, 1, 1)

    high = torch.quantile(
        flat, 0.99, dim=1
    ).view(-1, 1, 1, 1)

    denom = (
        high - threshold
    ).clamp_min(1e-6)

    residual = (
        (diff - threshold) / denom
    ).clamp(0.0, 1.0)

    residual = gaussian_blur_tensor(
        residual, blur_sigma
    ).clamp(0.0, 1.0)

    _check_finite("residual_map", residual)
    return residual


def proposed_forward(sr_model, lr, return_maps=False):
    """
    RRDBNet convolution은 AMP 사용.
    Depth / quantile / structure / blur / fusion은 FP32.
    최종 I_final에서 RRDBNet까지 gradient가 흐름.
    """

    # --------------------------------------------------------
    # 1. RRDBNet만 AMP
    # --------------------------------------------------------
    with torch.amp.autocast(
        device_type="cuda",
        enabled=AMP_ENABLED,
    ):
        I_sr = sr_model(lr)

    _check_finite("rrdbnet_output", I_sr)

    # 여기부터 FP32
    I_sr_fp32 = I_sr.float()

    I_up = F.interpolate(
        lr.float(),
        scale_factor=SCALE,
        mode="bicubic",
        align_corners=False,
    )

    _check_finite("bicubic_up", I_up)

    # --------------------------------------------------------
    # 2. Depth
    # --------------------------------------------------------
    M_depth = depth_map_from_lr(
        lr,
        I_sr.shape[-2:],
    ).to(
        device=I_sr.device,
        dtype=torch.float32,
    )

    # --------------------------------------------------------
    # 3. Edge
    # --------------------------------------------------------
    M_edge = make_edge_structure_map_torch(
        I_up,
        blur_sigma=1.5,
    )

    # --------------------------------------------------------
    # 4. Residual
    # --------------------------------------------------------
    M_residual = make_residual_structure_map_torch(
        I_up,
        I_sr_fp32,
        percentile=RESIDUAL_PERCENTILE,
        blur_sigma=2.0,
    )

    S = (
        EDGE_WEIGHT * M_edge
        + RESIDUAL_WEIGHT * M_residual
    ).clamp(0.0, 1.0)

    _check_finite("structure_map", S)

    # --------------------------------------------------------
    # 5. Depth -> far
    # --------------------------------------------------------
    D = M_depth.clamp(0.0, 1.0)

    far_strength = 1.0 - (
        (D - DEPTH_FAR)
        / max(DEPTH_NEAR - DEPTH_FAR, 1e-6)
    ).clamp(0.0, 1.0)

    far_strength = (
        far_strength
        * far_strength
        * (3.0 - 2.0 * far_strength)
    )

    _check_finite("far_strength", far_strength)

    # --------------------------------------------------------
    # 6. Structure protection
    # --------------------------------------------------------
    structure_raw = (
    (S - STRUCTURE_START)
    / max(1.0 - STRUCTURE_START, 1e-6)
    ).clamp(0.0, 1.0)

    POWER_EPS = 1e-4

    eps_pow = POWER_EPS ** STRUCTURE_PROTECT_POWER

    structure_protect = (
        (structure_raw + POWER_EPS).pow(
            STRUCTURE_PROTECT_POWER
        )
        - eps_pow
    )

    structure_protect = structure_protect / (
        (1.0 + POWER_EPS) ** STRUCTURE_PROTECT_POWER
        - eps_pow
    )

    structure_protect = structure_protect.clamp(
        0.0,
        1.0
    )

    _check_finite(
        "structure_protect",
        structure_protect
    )

    _check_finite(
        "structure_protect",
        structure_protect,
    )

    # --------------------------------------------------------
    # 7. Blur strength
    # --------------------------------------------------------
    blur_strength = far_strength * (
        1.0
        - (1.0 - MIN_STRUCTURE_BLUR)
        * structure_protect
    )

    blur_strength = gaussian_blur_tensor(
        blur_strength,
        MASK_SIGMA,
    ).clamp(0.0, 1.0)

    _check_finite("blur_strength", blur_strength)

    # --------------------------------------------------------
    # 8. Proposed final
    # --------------------------------------------------------
    I_blur = gaussian_blur_tensor(
        I_sr_fp32,
        BLUR_SIGMA,
    )

    _check_finite("blurred_sr", I_blur)

    I_final = (
        (1.0 - blur_strength) * I_sr_fp32
        + blur_strength * I_blur
    )

    _check_finite("I_final", I_final)

    if return_maps:
        maps = {
            "I_sr": I_sr_fp32,
            "I_up": I_up,
            "depth": D,
            "edge": M_edge,
            "residual": M_residual,
            "structure": S,
            "far_strength": far_strength,
            "structure_protect": structure_protect,
            "blur_strength": blur_strength,
        }
        return I_final, maps

    return I_final



# ============================================================
# Image -> EXACT training-time proposed_forward(), save visuals
# ============================================================
image_name = os.path.splitext(os.path.basename(IMAGE_PATH))[0]
experiment_dir = os.path.join(SAVE_PATH, image_name)
original_dir = os.path.join(experiment_dir, "00_original_realesrgan")
model_only_dir = os.path.join(experiment_dir, "01_model_only")
idea_dir = os.path.join(experiment_dir, "02_idea_applied")
for directory in (experiment_dir, original_dir, model_only_dir, idea_dir):
    os.makedirs(directory, exist_ok=True)

hr_bgr = cv2.imread(IMAGE_PATH, cv2.IMREAD_COLOR)
if hr_bgr is None:
    raise FileNotFoundError(IMAGE_PATH)
hr_h, hr_w = hr_bgr.shape[:2]
if MAKE_LOW_RESOLUTION:
    # Match DIV2K training pipeline: bicubic x4 downsampling in RGB PIL
    lr_pil = Image.fromarray(cv2.cvtColor(hr_bgr, cv2.COLOR_BGR2RGB))
    lr_pil = lr_pil.resize((max(1, hr_w // DOWNSCALE), max(1, hr_h // DOWNSCALE)), Image.Resampling.BICUBIC)
    lr_rgb = np.asarray(lr_pil).copy()
    lr_bgr = cv2.cvtColor(lr_rgb, cv2.COLOR_RGB2BGR)
else:
    lr_bgr = hr_bgr.copy()
    lr_rgb = cv2.cvtColor(lr_bgr, cv2.COLOR_BGR2RGB)

h, w = lr_bgr.shape[:2]
lr_tensor = torch.from_numpy(lr_rgb.copy()).permute(2, 0, 1).unsqueeze(0).float().div(255.0).to(device)

with torch.inference_mode():
    original_sr_tensor = original_model(lr_tensor).float()
    final, maps = proposed_forward(sr_train_model, lr_tensor, return_maps=True)


def rgb_tensor_to_bgr(x):
    arr = (x[0].detach().float().clamp(0, 1).permute(1, 2, 0).cpu().numpy() * 255.0).round().astype(np.uint8)
    return cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)


def map_to_gray(x):
    z = x[0, 0].detach().float().clamp(0, 1).cpu().numpy()
    return (z * 255.0).round().astype(np.uint8)


def save_png(filename, img, directory=None):
    path = os.path.join(directory or experiment_dir, filename)
    if not cv2.imwrite(path, img):
        raise RuntimeError(f"Could not save {path}")
    return path

I_original = rgb_tensor_to_bgr(original_sr_tensor)
I_sr = rgb_tensor_to_bgr(maps["I_sr"])
I_final = rgb_tensor_to_bgr(final)
I_up = cv2.resize(lr_bgr, (I_sr.shape[1], I_sr.shape[0]), interpolation=cv2.INTER_LANCZOS4)
# Lanczos is for display only. Training used bicubic I_up for its maps.

save_png("01_lr_input.png", lr_bgr)
save_png("03_original_realesrgan.png", I_original, original_dir)
save_png("03_original_realesrgan.png", I_original)
save_png("02_lanczos.png", I_up)
# ① BEST 체크포인트의 원본 출력: 추가 blur/fusion 없음
save_png("model_only_no_blur.png", I_sr, model_only_dir)
save_png("03_realesrgan.png", I_sr)  # 기존 파일명 호환
save_png("04_depth_gray.png", map_to_gray(maps["depth"]))
save_png("06_edge_map.png", map_to_gray(maps["edge"]))
save_png("07_residual_map.png", map_to_gray(maps["residual"]))
save_png("08_structure_map.png", map_to_gray(maps["structure"]))
save_png("09_far_strength.png", map_to_gray(maps["far_strength"]))
save_png("10_structure_protection.png", map_to_gray(maps["structure_protect"]))
save_png("11_blur_weight.png", map_to_gray(maps["blur_strength"]))
with torch.inference_mode():
    blurred = gaussian_blur_tensor(maps["I_sr"], BLUR_SIGMA)
save_png("12_realesrgan_blurred.png", rgb_tensor_to_bgr(blurred))
# ② 동일 체크포인트 출력에 학습 당시의 Depth/Structure/Blur 아이디어 적용
save_png("idea_applied.png", I_final, idea_dir)
save_png("13_proposed.png", I_final)  # 기존 파일명 호환

diff = np.abs(I_sr.astype(np.float32) - I_final.astype(np.float32)).mean(axis=2)
np.save(os.path.join(experiment_dir, "14_difference_raw.npy"), diff)
scale = max(float(np.percentile(diff, 99)), 1e-8)
save_png("15_difference_gray.png", (np.clip(diff / scale, 0, 1) * 255).astype(np.uint8))

D = maps["depth"][0,0].cpu().numpy()
S = maps["structure"][0,0].cpu().numpy()
far = maps["far_strength"][0,0].cpu().numpy()
prot = maps["structure_protect"][0,0].cpu().numpy()
blur_weight = maps["blur_strength"][0,0].cpu().numpy()

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(D, cmap="inferno", vmin=0, vmax=1)
ax.set_title("Depth Anything V2 (0=Far, 1=Near)")
ax.axis("off")
fig.colorbar(im, ax=ax)
fig.tight_layout()
fig.savefig(os.path.join(experiment_dir, "05_depth_visualization.png"), dpi=200, bbox_inches="tight")
plt.close(fig)

fig, axes = plt.subplots(2, 4, figsize=(22, 11))
items = [
    (D, "Depth", "inferno"),
    (far, "Far Strength", "gray"),
    (S, "Structure", "gray"),
    (prot, "Structure Protection", "viridis"),
    (cv2.cvtColor(lr_bgr, cv2.COLOR_BGR2RGB), "LR Input", None),
    (cv2.cvtColor(I_sr, cv2.COLOR_BGR2RGB), "Fine-tuned RRDBNet", None),
    (blur_weight, "Blur Weight", "magma"),
    (cv2.cvtColor(I_final, cv2.COLOR_BGR2RGB), "Proposed (Best Checkpoint)", None),
]
for ax, (arr, title, cmap) in zip(axes.flat, items):
    ax.imshow(arr, cmap=cmap, vmin=0 if cmap else None, vmax=1 if cmap else None)
    ax.set_title(title)
    ax.axis("off")
fig.tight_layout()
fig.savefig(os.path.join(experiment_dir, "16_internal_visualization.png"), dpi=200, bbox_inches="tight")
plt.close(fig)

# Three-way comparison: all outputs from the EXACT same LR input
fig, axes = plt.subplots(1, 4, figsize=(24, 7))
comparison = [
    (lr_bgr, "Input LR"),
    (I_original, "1. Original Real-ESRGAN"),
    (I_sr, "2. Best Model Only"),
    (I_final, "3. Best Model + Idea"),
]
for ax, (im, title) in zip(axes, comparison):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=13)
    ax.axis("off")
fig.tight_layout()
fig.savefig(os.path.join(experiment_dir, "17_three_way_comparison.png"), dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, axes = plt.subplots(1, 4, figsize=(24, 6))
for ax, (im, title) in zip(axes[:3], comparison[1:]):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis("off")
plot = axes[3].imshow(diff, cmap="hot")
axes[3].set_title("Difference: Model Only vs Idea")
axes[3].axis("off")
fig.colorbar(plot, ax=axes[3], fraction=.046, pad=.04)
fig.tight_layout()
fig.savefig(os.path.join(experiment_dir, "18_three_results_and_difference.png"), dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

# ============================================================
# Additional visualizations: all internal maps and 3-way outputs
# ============================================================
visual_dir = os.path.join(experiment_dir, "visualizations")
os.makedirs(visual_dir, exist_ok=True)

# Use maps generated by the exact training-time proposed_forward.
map_specs = [
    ("depth", "Depth Map (0=Far, 1=Near)", "inferno"),
    ("far_strength", "Far Strength", "gray"),
    ("edge", "Edge Map", "gray"),
    ("residual", "Residual Structure Map", "gray"),
    ("structure", "Structure Map", "gray"),
    ("structure_protect", "Structure Protection", "viridis"),
    ("blur_strength", "Final Blur Weight", "magma"),
]

# Individual heatmaps include a shared numeric range (0~1) and colorbar.
for idx, (key, title, cmap) in enumerate(map_specs, start=1):
    arr = maps[key][0, 0].detach().float().cpu().numpy()
    fig, ax = plt.subplots(figsize=(10, 7))
    im = ax.imshow(arr, cmap=cmap, vmin=0, vmax=1)
    ax.set_title(title, fontsize=15)
    ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(
        os.path.join(visual_dir, f"{idx:02d}_{key}_heatmap.png"),
        dpi=200, bbox_inches="tight"
    )
    plt.close(fig)

# Compare the inner process in a detailed 3x4 overview.
with torch.inference_mode():
    blurred_display = rgb_tensor_to_bgr(
        gaussian_blur_tensor(maps["I_sr"], BLUR_SIGMA)
    )

panels = [
    (lr_bgr, "Input LR", None, True),
    (I_original, "Original Real-ESRGAN", None, True),
    (I_sr, "Best Model Only", None, True),
    (I_final, "Best Model + Idea", None, True),
    (D, "Depth (0 Far / 1 Near)", "inferno", False),
    (far, "Far Strength", "gray", False),
    (maps["edge"][0,0].cpu().numpy(), "Edge Map", "gray", False),
    (maps["residual"][0,0].cpu().numpy(), "Residual Map", "gray", False),
    (S, "Structure Map", "gray", False),
    (prot, "Structure Protection", "viridis", False),
    (blur_weight, "Blur Weight", "magma", False),
    (diff, "Difference (Model vs Idea)", "hot", False),
]
fig, axes = plt.subplots(3, 4, figsize=(25, 18))
for ax, (arr, title, cmap, is_bgr) in zip(axes.flat, panels):
    if is_bgr:
        ax.imshow(cv2.cvtColor(arr, cv2.COLOR_BGR2RGB))
    else:
        # Difference map retains its native intensity in 0-255 units.
        im = ax.imshow(arr, cmap=cmap, **({} if title.startswith("Difference") else {"vmin":0,"vmax":1}))
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(title, fontsize=13)
    ax.axis("off")
fig.tight_layout()
fig.savefig(os.path.join(experiment_dir, "20_full_internal_visualization.png"), dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

# Highlight how the fine-tuned model is changed by the idea.
fig, axes = plt.subplots(2, 3, figsize=(21, 13))
additional_panels = [
    (I_sr, "Best Model Only", None, True),
    (blurred_display, "Gaussian Blurred Model Output", None, True),
    (I_final, "Best Model + Idea", None, True),
    (D, "Depth", "inferno", False),
    (blur_weight, "Adaptive Blur Weight", "magma", False),
    (diff, "Absolute Difference", "hot", False),
]
for ax, (arr, title, cmap, is_bgr) in zip(axes.flat, additional_panels):
    if is_bgr:
        ax.imshow(cv2.cvtColor(arr, cv2.COLOR_BGR2RGB))
    else:
        kwargs = {} if title == "Absolute Difference" else {"vmin": 0, "vmax": 1}
        im = ax.imshow(arr, cmap=cmap, **kwargs)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(title, fontsize=13)
    ax.axis("off")
fig.tight_layout()
fig.savefig(os.path.join(experiment_dir, "21_idea_process_visualization.png"), dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Detailed visualizations saved:", visual_dir)
print("Full internal visualization:", os.path.join(experiment_dir, "20_full_internal_visualization.png"))
print("Idea process visualization:", os.path.join(experiment_dir, "21_idea_process_visualization.png"))

far_region = far >= 0.5
structured_region = prot >= 0.5
stats = {
    "IMAGE_PATH": IMAGE_PATH,
    "BEST_PATH": BEST_PATH,
    "ORIGINAL_PATH": ORIGINAL_PATH,
    "BEST_EPOCH": ckpt.get("epoch", "unknown"),
    "BEST_VAL_PSNR": ckpt.get("val_psnr", "unknown"),
    "LR_SIZE": f"{w}x{h}",
    "SR_SIZE": f"{I_sr.shape[1]}x{I_sr.shape[0]}",
    "DEPTH_FAR": DEPTH_FAR,
    "DEPTH_NEAR": DEPTH_NEAR,
    "STRUCTURE_START": STRUCTURE_START,
    "STRUCTURE_PROTECT_POWER": STRUCTURE_PROTECT_POWER,
    "MIN_STRUCTURE_BLUR": MIN_STRUCTURE_BLUR,
    "BLUR_SIGMA": BLUR_SIGMA,
    "MASK_SIGMA": MASK_SIGMA,
    "depth_mean": float(D.mean()),
    "structure_mean": float(S.mean()),
    "far_strength_mean": float(far.mean()),
    "blur_weight_mean": float(blur_weight.mean()),
    "difference_mean": float(diff.mean()),
    "far_structure_blur_mean": float(blur_weight[far_region & structured_region].mean()) if np.any(far_region & structured_region) else "N/A",
    "far_no_structure_blur_mean": float(blur_weight[far_region & ~structured_region].mean()) if np.any(far_region & ~structured_region) else "N/A",
}
with open(os.path.join(experiment_dir, "19_parameters_and_stats.txt"), "w", encoding="utf-8") as f:
    for k, v in stats.items(): f.write(f"{k}={v}\n")

print("EXPERIMENT COMPLETE")
print("Loaded checkpoint:", BEST_PATH)
print("Saved to:", experiment_dir)
print("Original Real-ESRGAN:", os.path.join(original_dir, "03_original_realesrgan.png"))
print("Model only (NO additional blur):", os.path.join(model_only_dir, "model_only_no_blur.png"))
print("Idea applied (Depth + Structure + Blur):", os.path.join(idea_dir, "idea_applied.png"))
print("Three-way comparison:", os.path.join(experiment_dir, "17_three_way_comparison.png"))
print("Proposed output:", os.path.join(experiment_dir, "13_proposed.png"))
print("Difference comparison:", os.path.join(experiment_dir, "18_three_results_and_difference.png"))
